In [1]:
from curl_cffi import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import time
import re
import json
from datetime import datetime
import unicodedata

# ================== CONFIG ==================
REQUEST_DELAY = 1.0
MAX_HOT_PAGES = 30
HOT_PAGE_BASE = "https://spectator.org/hot-off-the-press"
OUTPUT_FILE = "spectator_articles.json"
OUTLET_NAME = "The American Spectator"
TARGET_ARTICLES = 200
MIN_WORD_COUNT = 150  # Minimum words for a valid article

PROMO_REGEX = re.compile(
    r"^(read more\b.*|related:.*|\(related:.*)",
    re.IGNORECASE
)

BLOCKED_PATHS = [
    "/register",
    "/login",
    "/author",
    "/tag",
    "/category",
    "/wp-content",
    "/wp-json",
    "/subscribe",
    "/about",
    "/privacy-policy",
    "/contact",
    "/magazine",
    "/don-lemon-arrested",
]
BLOCKED_REGEX = [
    r"-ep-\d+"   # podcast episodes
]
BLOCKED_DOMAINS = [
    "twitter.com",
    "x.com",
    "facebook.com",
    "instagram.com",
    "linkedin.com",
    "youtube.com",
    "youtu.be",
]
IMPERSONATE = "chrome120"

# ================== HELPERS ==================
def normalize_unicode(text):
    if not text:
        return text

    text = unicodedata.normalize("NFC", text)

    replacements = {
        "\u201c": '"',
        "\u201d": '"',
        "\u2018": "'",
        "\u2019": "'",
        "\u02BC": "'",
        "\u2032": "'",
        "\u2013": "-",
        "\u2014": "-",
        "\u2026": "...",
        "\u00a0": " ",
        "\u2029": " ",   # paragraph separator
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    # Remove zero-width characters
    text = re.sub(r"[\u200B-\u200D\uFEFF]", "", text)

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

def utc_now_iso():
    return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"

def extract_slug(url):
    path = urlparse(url).path.rstrip("/")
    return path.split("/")[-1] if path else None

def is_blocked(url):
    parsed = urlparse(url)
    path = parsed.path.lower()
    domain = parsed.netloc.lower()
    full_url = url.lower()
    
    for blocked in BLOCKED_DOMAINS:
        if domain == blocked or domain.endswith("." + blocked):
            return True
    
    for blocked in BLOCKED_PATHS:
        if path.startswith(blocked):
            return True
    
    for pattern in BLOCKED_REGEX:
        if re.search(pattern, full_url):
            return True
    
    return False

def fetch_page(url, timeout=10):
    """Fetch page with browser impersonation"""
    response = requests.get(
        url, 
        impersonate=IMPERSONATE,
        timeout=timeout
    )
    response.raise_for_status()
    return response.text

def count_words(text):
    """Count words in text"""
    if not text:
        return 0
    # Split by whitespace and count
    words = text.split()
    return len(words)

# ================== ARTICLE EXTRACTION ==================
def extract_article_data(url):
    """Extract article metadata and content"""
    print(f"  Extracting from: {url}")
    
    try:
        html = fetch_page(url)
        soup = BeautifulSoup(html, "html.parser")
        
        article_id = extract_slug(url)
        
        # ================== HEADLINE ==================
        headline = None
        title_div = soup.find("div", class_="title pf-title")
        if title_div:
            headline = title_div.get_text(separator=" ", strip=True)
        else:
            h1_tag = soup.find("h1", class_="entry-title")
            if h1_tag:
                headline = h1_tag.get_text(separator=" ", strip=True)

        if headline:
            headline = normalize_unicode(headline)
        
        crawl_timestamp = utc_now_iso()

        # ================== DATE ==================
        publication_date = None
        date_div = soup.find("div", class_="date print-only")
        if date_div:
            publication_date = date_div.get_text(strip=True)
        else:
            time_tag = soup.find("time", class_="entry-date")
            if time_tag and time_tag.get("datetime"):
                publication_date = time_tag.get("datetime")
        
        # ================== BODY ==================
        body_text = None
        post_body = soup.find("div", class_="post-body print-only")
        if not post_body:
            post_body = soup.find("div", class_="post-body")
        
        if post_body:
            paywall = post_body.find("div", class_="paywall")
            
            if paywall:
                # Remove scripts, styles, ads, iframes
                for el in paywall.find_all(['script', 'style', 'ins', 'iframe']):
                    el.decompose()
                
                # Remove ad / disclaimer divs
                for div in paywall.find_all('div'):
                    classes = div.get('class', [])
                    if any('ad' in c.lower() or 'spect-' in c or 'disclaimer' in c for c in classes):
                        div.decompose()
                
                # Remove captions
                for p in paywall.find_all('p', class_='wp-caption-text'):
                    p.decompose()
                for p in paywall.find_all('p', id=re.compile(r'caption-attachment-')):
                    p.decompose()
                
                # Remove author bio paragraphs
                for p in paywall.find_all('p'):
                    if p.find('a', href=re.compile(r'^mailto:', re.I)):
                        p.decompose()
                        continue

                    t = p.get_text(separator=' ', strip=True).lower()
                    if (
                        'write to him at' in t
                        or 'write to her at' in t
                        or 'write to them at' in t
                        or ('is a' in t and 'director' in t)
                    ):
                        p.decompose()
                
                # ================== REMOVE INLINE (RELATED: ...) ==================
                for strong in paywall.find_all('strong'):
                    t = strong.get_text(separator=' ', strip=True).lower()
                    if '(related:' in t:
                        strong.decompose()
                
                # Unwrap drop-cap spans
                for span in paywall.find_all('span', class_='first-char'):
                    span.unwrap()
                
                # ================== TEXT EXTRACTION ==================
                text_parts = []
                for el in paywall.find_all([
                    'p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'blockquote', 'li'
                ]):
                    text = el.get_text(separator=' ', strip=True)

                    # Normalize whitespace
                    text = re.sub(r'\s+', ' ', text).strip()
                    
                    # Remove "READ MORE…", "RELATED…", inline promo blocks
                    if PROMO_REGEX.match(text.lower()):
                        continue
                    
                    # Remove trailing author promo headers
                    if text.lower().startswith(("image licensed", "image credit")):
                        continue
                    
                    # Fix split drop-caps (e.g., "I left" → "Ileft")
                    text = re.sub(r'^([A-Z])\s+([a-z])', r'\1\2', text)
                    
                    if text and len(text.split()) >= 6:
                        text_parts.append(text)
                
                body_text = '\n\n'.join(text_parts)
                body_text = normalize_unicode(body_text).strip()
        
        # ================== WORD COUNT ==================
        word_count = count_words(body_text)
        if word_count < MIN_WORD_COUNT:
            print(f"  ⚠ Skipping - only {word_count} words (minimum: {MIN_WORD_COUNT})")
            return None
        
        article = {
            "article_id": article_id,
            "headline": headline,
            "body_text": body_text,
            "publication_date": publication_date,
            "crawl_time": crawl_timestamp,
            "outlet_name": OUTLET_NAME,
            "word_count": word_count,
            "label": "right",
            "url": url
        }
        
        print(f"  ✓ Extracted: {headline[:50]}... ({word_count} words)")
        return article
    
    except Exception as e:
        print(f"  ✗ Error: {e}")
        return None
        
# ================== HOT-OFF-THE-PRESS CRAWLER ==================
def crawl_hot_off_the_press():
    articles = []
    seen_slugs = set()
    
    for page in range(1, MAX_HOT_PAGES + 1):
        url = HOT_PAGE_BASE if page == 1 else f"{HOT_PAGE_BASE}/page/{page}"
        print(f"Fetching: {url}")
        
        try:
            html = fetch_page(url)
        except Exception as e:
            print(f"Stopped: {e}")
            break
        
        soup = BeautifulSoup(html, "html.parser")
        main_loop = soup.find("div", class_="main-loop")
        
        if not main_loop:
            print("No main-loop found, stopping.")
            break
        
        found_any = False
        for post in main_loop.find_all("div", class_="large-post"):
            a = post.find("a", href=True)
            if not a:
                continue
            
            href = a["href"]
            if is_blocked(href):
                continue
            
            slug = extract_slug(href)
            if not slug or slug in seen_slugs:
                continue
            
            seen_slugs.add(slug)
            articles.append(href)
            found_any = True

        if not found_any:
            print("No new articles found on this page, stopping.")
            break
        
        time.sleep(REQUEST_DELAY)
    
    return articles

# ================== RUN ==================
if __name__ == "__main__":
    print(f"{'='*60}")
    print("SPECTATOR ARTICLE SCRAPER")
    print(f"{'='*60}\n")
    
    # Crawl article URLs
    hot_articles = crawl_hot_off_the_press()
    
    print(f"\n{'='*60}")
    print(f"Found {len(hot_articles)} article URLs")
    print(f"{'='*60}\n")
    
    # Extract content from each article
    all_articles = []

    for i, url in enumerate(hot_articles, 1):
        if len(all_articles) >= TARGET_ARTICLES:
            break
    
        print(f"[{i}/{len(hot_articles)}]")
        article_data = extract_article_data(url)
    
        if article_data:
            all_articles.append(article_data)
    
        if i < len(hot_articles):
            time.sleep(REQUEST_DELAY)
    
    # Save to JSON
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(all_articles, f, ensure_ascii=False, indent=2)
    
    # Summary
    print(f"\n{'='*60}")
    print("EXTRACTION SUMMARY")
    print(f"{'='*60}")
    successful = len(all_articles)
    total_words = sum(a.get('word_count', 0) for a in all_articles)
    avg_words = total_words / len(all_articles) if all_articles else 0
    
    print(f"Total URLs processed: {len(hot_articles)}")
    print(f"Valid articles saved: {successful}")
    print(f"Skipped (too short or errors): {len(hot_articles) - successful}")
    print(f"Total words: {total_words:,}")
    print(f"Average words per article: {avg_words:.0f}")
    print(f"\nData saved to: {OUTPUT_FILE}")
    
    # Show sample
    if all_articles and all_articles[0].get('headline'):
        print(f"\n{'='*60}")
        print("SAMPLE ARTICLE")
        print(f"{'='*60}")
        sample = all_articles[0]
        print(f"Article ID: {sample['article_id']}")
        print(f"Headline: {sample['headline']}")
        print(f"Publication Date: {sample['publication_date']}")
        print(f"Outlet: {sample['outlet_name']}")
        print(f"Word Count: {sample['word_count']:,}")
        print(f"Body Text Length: {len(sample['body_text']) if sample['body_text'] else 0} characters")
        if sample['body_text']:
            print(f"\nBody preview (first 300 chars):\n{sample['body_text'][:300]}...")

SPECTATOR ARTICLE SCRAPER

Fetching: https://spectator.org/hot-off-the-press
Fetching: https://spectator.org/hot-off-the-press/page/2
Fetching: https://spectator.org/hot-off-the-press/page/3
Fetching: https://spectator.org/hot-off-the-press/page/4
Fetching: https://spectator.org/hot-off-the-press/page/5
Fetching: https://spectator.org/hot-off-the-press/page/6
Fetching: https://spectator.org/hot-off-the-press/page/7
Fetching: https://spectator.org/hot-off-the-press/page/8
Fetching: https://spectator.org/hot-off-the-press/page/9
Fetching: https://spectator.org/hot-off-the-press/page/10
Fetching: https://spectator.org/hot-off-the-press/page/11
Fetching: https://spectator.org/hot-off-the-press/page/12
Fetching: https://spectator.org/hot-off-the-press/page/13
Fetching: https://spectator.org/hot-off-the-press/page/14
Fetching: https://spectator.org/hot-off-the-press/page/15
Fetching: https://spectator.org/hot-off-the-press/page/16
Fetching: https://spectator.org/hot-off-the-press/page/17
Fet